# 경제 분석 및 예측과 데이터 지능 실습 3.1: 시계열 데이터 엔지니어링

## 이번 실습에서 다루는 것

- 시계열 컬럼 타입 정리(`datetime`, 카테고리, 인덱스)
- 결측/중복/빈도 점검
- `groupby` + `agg`를 이용한 다양한 집계
- 일별 데이터의 월별/주별 리샘플링(`resample`)
- 정규화/스케일링/로그/증감률 피처 생성
- `melt`, `pivot_table`을 이용한 long/wide 구조 변환

## 사용 데이터

- 파일: `bike_sharing_daily.csv`
- 단위: 일별 관측치
- 주요 변수: `cnt`, `casual`, `registered`, `temp`, `hum`, `windspeed`, `weathersit`

References:
- [Pandas Time Series](https://pandas.pydata.org/docs/user_guide/timeseries.html)
- [Pandas GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Pandas Reshaping](https://pandas.pydata.org/docs/user_guide/reshaping.html)

In [ ]:
# 실행 환경 설정
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [13]:
# 범주형 변수 해석용 매핑
season_map = {1: "spring", 2: "summer", 3: "fall", 4: "winter"}
weather_map = {1: "clear", 2: "mist_cloud", 3: "light_rain_snow", 4: "heavy_rain_snow"}

print("Mappings loaded.")

Mappings loaded.


In [30]:
# 데이터 로드 및 기본 확인
df_raw = pd.read_csv("../datasets/bike_sharing_daily.csv")

print("raw shape:", df_raw.shape)
print("columns:", list(df_raw.columns))
# dteday의 자료형
print(type(df_raw["dteday"][0]))
df_raw.head()

raw shape: (731, 16)
columns: ['instant', 'dteday', 'season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
<class 'str'>


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.3442,0.3636,0.8058,0.1604,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.3635,0.3537,0.6961,0.2485,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.1964,0.1894,0.4373,0.2483,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.2000,0.2121,0.5904,0.1603,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.2270,0.2293,0.4370,0.1869,82,1518,1600


## 0) 시계열 타입 처리와 데이터 품질 점검

모델링 전 가장 먼저 해야 할 일은 **시간축 정합성 확인**입니다.

### 체크 항목

- `datetime`으로 변환했는가?
- 날짜 정렬이 되어 있는가?
- 중복 날짜가 있는가?
- 날짜 빈도(일별)가 끊긴 구간은 없는가?
- 범주형 변수(`season`, `weathersit`)를 해석 가능한 라벨로 바꿨는가?

아래 셀에서는 위 항목을 한 번에 점검하고, 이후 집계에 바로 쓰일 `df_daily`를 만듭니다.

In [33]:
df_daily = df_raw.copy()

df_daily["dteday"] = pd.to_datetime(df_daily["dteday"])

df_daily = df_daily.sort_values("dteday").set_index("dteday")

df_daily.index.name = "date"

In [43]:
df_daily["season_name"] = df_daily["season"].map(season_map).astype("category")
df_daily["weather_name"] = df_daily["weathersit"].map(weather_map).astype("category")

df_daily["year"] = df_daily.index.year
df_daily["month"] = df_daily.index.month
df_daily["year_month"] = df_daily.index.to_period("M").astype(str)

print("date range:", df_daily.index.min().date(), "~", df_daily.index.max().date())
print("duplicated dates:", int(df_daily.index.duplicated().sum()))
print("missing daily stamps:", int(df_daily.asfreq("D").index.difference(df_daily.index).shape[0]))
print("shape:", df_daily.shape)

display(df_daily[["year_month","cnt", "casual", "registered", "temp", "hum", "weather_name"]].head())

date range: 2011-01-01 ~ 2012-12-31
duplicated dates: 0
missing daily stamps: 0
shape: (731, 20)


,year_month,cnt,casual,registered,temp,hum,weather_name
date,,,,,,,
2011-01-01,2011-01,985,331,654,0.3442,0.8058,mist_cloud
2011-01-02,2011-01,801,131,670,0.3635,0.6961,mist_cloud
2011-01-03,2011-01,1349,120,1229,0.1964,0.4373,clear
2011-01-04,2011-01,1562,108,1454,0.2000,0.5904,clear
2011-01-05,2011-01,1600,82,1518,0.2270,0.4370,clear


## 1) GroupBy + Aggregate: 다양한 집계

`groupby`는 **무엇으로 묶고(그룹 키), 무엇을 계산할지(집계 함수)** 를 목적에 맞게 선택하는 것이 중요.

이번 파트에서는 아래를 모두 연습합니다.

1. 연/월 단위 수요 집계
2. 요일별 평균/분산 집계
3. 계절 × 날씨 조합 집계
4. 사용자 유형(`casual`, `registered`) 비율 집계
5. 사용자 정의 지표(주말 비중, 피크 대비 평균 등)

In [46]:
df_daily.head(5)

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt,season_name,weather_name,year,month,year_month
date,,,,,,,,,,,,,,,,,,,,
2011-01-01,1,1,0,1,0,6,0,2,0.3442,0.3636,0.8058,0.1604,331,654,985,spring,mist_cloud,2011,1,2011-01
2011-01-02,2,1,0,1,0,0,0,2,0.3635,0.3537,0.6961,0.2485,131,670,801,spring,mist_cloud,2011,1,2011-01
2011-01-03,3,1,0,1,0,1,1,1,0.1964,0.1894,0.4373,0.2483,120,1229,1349,spring,clear,2011,1,2011-01
2011-01-04,4,1,0,1,0,2,1,1,0.2000,0.2121,0.5904,0.1603,108,1454,1562,spring,clear,2011,1,2011-01
2011-01-05,5,1,0,1,0,3,1,1,0.2270,0.2293,0.4370,0.1869,82,1518,1600,spring,clear,2011,1,2011-01


In [47]:
# 월별 집계: 수요 합계/평균, 기온/습도 평균
monthly_summary = (
    df_daily.groupby(["year", "month"], as_index=False)
    .agg(
        cnt_sum=("cnt", "sum"),
        cnt_mean=("cnt", "mean"),
        cnt_std=("cnt", "std"),
        temp_mean=("temp", "mean"),
        hum_mean=("hum", "mean"),
        wind_mean=("windspeed", "mean")
    )
)

# 요일별 집계
weekday_summary = (
    df_daily.groupby("weekday", as_index=False)
    .agg(
        cnt_mean=("cnt", "mean"),
        cnt_median=("cnt", "median"),
        cnt_std=("cnt", "std"),
        casual_mean=("casual", "mean"),
        registered_mean=("registered", "mean")
    )
    .sort_values("weekday")
)

print("[monthly_summary head]")
display(monthly_summary.head(8))
print("[weekday_summary]")
display(weekday_summary)

[monthly_summary head]


,year,month,cnt_sum,cnt_mean,cnt_std,temp_mean,hum_mean,wind_mean
0,2011,1,38189,"1,231.9032",372.4327,0.1977,0.5844,0.1954
1,2011,2,48215,"1,721.9643",398.5088,0.2825,0.5601,0.2286
2,2011,3,64045,"2,065.9677",550.9717,0.3317,0.5694,0.2324
3,2011,4,94870,"3,162.3333","1,042.0936",0.4712,0.6683,0.2442
4,2011,5,135821,"4,381.3226",572.9279,0.5772,0.7134,0.1813
5,2011,6,143512,"4,783.7333",444.4478,0.6931,0.5932,0.1782
6,2011,7,141341,"4,559.3871",680.0908,0.7586,0.5897,0.1717
7,2011,8,136691,"4,409.3871",809.8524,0.7054,0.6268,0.1907


[weekday_summary]


,weekday,cnt_mean,cnt_median,cnt_std,casual_mean,registered_mean
0,0,"4,228.8286","4,334.0000","1,872.4966","1,338.2952","2,890.5333"
1,1,"4,338.1238","4,359.0000","1,793.0740",674.1333,"3,663.9905"
2,2,"4,510.6635","4,576.5000","1,826.9116",556.1827,"3,954.4808"
3,3,"4,548.5385","4,642.5000","2,038.0959",551.1442,"3,997.3942"
4,4,"4,667.2596","4,721.0000","1,939.4333",590.9615,"4,076.2981"
5,5,"4,690.2885","4,601.5000","1,874.6249",752.2885,"3,938.0000"
6,6,"4,550.5429","4,521.0000","2,196.6930","1,465.2571","3,085.2857"


In [48]:
monthly_summary2 = (
    df_daily.groupby(["month", "year"], as_index=False)
    .agg(
        cnt_sum=("cnt", "sum"),
        cnt_mean=("cnt", "mean"),
        cnt_std=("cnt", "std"),
        temp_mean=("temp", "mean"),
        hum_mean=("hum", "mean"),
        wind_mean=("windspeed", "mean")
    )
)

monthly_summary2.head(8)

,month,year,cnt_sum,cnt_mean,cnt_std,temp_mean,hum_mean,wind_mean
0,1,2011,38189,"1,231.9032",372.4327,0.1977,0.5844,0.1954
1,1,2012,96744,"3,120.7742",872.8521,0.2752,0.5873,0.2172
2,2,2011,48215,"1,721.9643",398.5088,0.2825,0.5601,0.2286
3,2,2012,103137,"3,556.4483",870.7246,0.3153,0.5746,0.2032
4,3,2011,64045,"2,065.9677",550.9717,0.3317,0.5694,0.2324
5,3,2012,164875,"5,318.5484","1,251.1627",0.4494,0.6075,0.2130
6,4,2011,94870,"3,162.3333","1,042.0936",0.4712,0.6683,0.2442
7,4,2012,174224,"5,807.4667","1,308.9389",0.4688,0.5078,0.2247


In [49]:
# 계절 x 날씨 집계 + 사용자 유형 비율 계산
season_weather_summary = (
    df_daily.groupby(["season_name", "weather_name"], observed=True, as_index=False)
    .agg(
        cnt_mean=("cnt", "mean"),
        cnt_q25=("cnt", lambda s: s.quantile(0.25)),
        cnt_q75=("cnt", lambda s: s.quantile(0.75)),
        casual_mean=("casual", "mean"),
        registered_mean=("registered", "mean")
    )
)

season_weather_summary["casual_ratio"] = (
    season_weather_summary["casual_mean"]
    / (season_weather_summary["casual_mean"] + season_weather_summary["registered_mean"])
)

print("[season_weather_summary]")
display(season_weather_summary.sort_values(["season_name", "weather_name"]))

[season_weather_summary]


,season_name,weather_name,cnt_mean,cnt_q25,cnt_q75,casual_mean,registered_mean,casual_ratio
0,fall,clear,"5,878.2574","4,688.5000","7,121.0000","1,234.8897","4,643.3676",0.2101
1,fall,light_rain_snow,"2,751.7500","1,957.5000","3,147.2500",434.7500,"2,317.0000",0.1580
2,fall,mist_cloud,"5,222.4792","4,152.2500","6,292.2500","1,175.1458","4,047.3333",0.2250
3,spring,clear,"2,811.1351","1,629.0000","3,893.0000",374.4324,"2,436.7027",0.1332
4,spring,light_rain_snow,934.7500,489.7500,"1,009.5000",70.2500,864.5000,0.0752
5,spring,mist_cloud,"2,357.1667","1,381.5000","3,052.2500",284.5303,"2,072.6364",0.1207
6,summer,clear,"5,548.5487","4,595.0000","6,734.0000","1,295.4690","4,253.0796",0.2335
7,summer,light_rain_snow,"1,169.0000",911.0000,"1,356.0000",140.0000,"1,029.0000",0.1198
8,summer,mist_cloud,"4,236.7059","3,211.5000","5,151.2500",834.0294,"3,402.6765",0.1969
9,winter,clear,"5,043.5631","3,797.0000","5,919.5000",878.1650,"4,165.3981",0.1741


## 2) 시계열 리샘플링: 월별/주별 집계

일별 데이터는 그대로 쓰기보다, 분석 목적에 따라 주/월 단위로 재집계하는 경우가 많습니다.

### 핵심 연산

- `resample("ME")`: 월말 기준 집계
- `resample("W")`: 주간 집계
- `asfreq()`: 빈도 강제 정렬(결측 탐지용)

In [18]:
monthly_ts = (
    df_daily[["cnt", "casual", "registered"]]
    .resample("ME")
    .sum()
)
monthly_ts["cnt_ma3"] = monthly_ts["cnt"].rolling(3, min_periods=1).mean()
monthly_ts["casual_share"] = monthly_ts["casual"] / monthly_ts["cnt"]

weekly_ts = (
    df_daily[["cnt", "temp", "hum", "windspeed"]]
    .resample("W")
    .agg({"cnt": "sum", "temp": "mean", "hum": "mean", "windspeed": "mean"})
)

print("[monthly_ts head]")
display(monthly_ts.head(8))
print("[weekly_ts head]")
display(weekly_ts.head(8))

[monthly_ts head]


,cnt,casual,registered,cnt_ma3,casual_share
date,,,,,
2011-01-31,38189,3073,35116,"38,189.0000",0.0805
2011-02-28,48215,6242,41973,"43,202.0000",0.1295
2011-03-31,64045,12826,51219,"50,149.6667",0.2003
2011-04-30,94870,22346,72524,"69,043.3333",0.2355
2011-05-31,135821,31050,104771,"98,245.3333",0.2286
2011-06-30,143512,30612,112900,"124,734.3333",0.2133
2011-07-31,141341,36452,104889,"140,224.6667",0.2579
2011-08-31,136691,28842,107849,"140,514.6667",0.2110


[weekly_ts head]


,cnt,temp,hum,windspeed
date,,,,
2011-01-02,1786,0.3538,0.7510,0.2045
2011-01-09,9408,0.1896,0.4931,0.2118
2011-01-16,9025,0.1834,0.5371,0.2034
2011-01-23,8770,0.1828,0.5675,0.2167
2011-01-30,7699,0.1928,0.6894,0.1484
2011-02-06,10273,0.2215,0.6756,0.1733
2011-02-13,11192,0.2142,0.5309,0.1987
2011-02-20,14692,0.3773,0.3902,0.3124


## 3) 피처 엔지니어링

- `z-score` 표준화
- `min-max` 정규화
- 로그 변환
- 1일/7일 lag
- 일간 증감률(`pct_change`)

주의: lag/증감률은 앞부분 결측을 만들기 때문에 `dropna()` 시점도 의도적으로 관리해야 합니다.

In [19]:
engineered = df_daily[["cnt", "temp", "hum", "windspeed", "casual", "registered"]].copy()

for col in ["cnt", "temp", "hum", "windspeed"]:
    engineered[f"{col}_z"] = (engineered[col] - engineered[col].mean()) / engineered[col].std()
    engineered[f"{col}_minmax"] = (engineered[col] - engineered[col].min()) / (engineered[col].max() - engineered[col].min())

engineered["cnt_log1p"] = np.log1p(engineered["cnt"])
engineered["cnt_lag1"] = engineered["cnt"].shift(1)
engineered["cnt_lag7"] = engineered["cnt"].shift(7)
engineered["cnt_pct_change"] = engineered["cnt"].pct_change()
engineered["temp_diff1"] = engineered["temp"].diff(1)

print("engineered shape:", engineered.shape)
display(engineered.head(10))

engineered shape: (731, 19)


,cnt,temp,hum,windspeed,casual,registered,cnt_z,cnt_minmax,temp_z,temp_minmax,hum_z,hum_minmax,windspeed_z,windspeed_minmax,cnt_log1p,cnt_lag1,cnt_lag7,cnt_pct_change,temp_diff1
date,,,,,,,,,,,,,,,,,,,
2011-01-01,985,0.3442,0.8058,0.1604,331,654,-1.8167,0.1108,-0.8261,0.3552,1.2493,0.8286,-0.3876,0.2846,6.8937,NaN,NaN,NaN,NaN
2011-01-02,801,0.3635,0.6961,0.2485,131,670,-1.9117,0.0896,-0.7206,0.3792,0.4788,0.7158,0.7491,0.4662,6.6871,985.0000,NaN,-0.1868,0.0193
2011-01-03,1349,0.1964,0.4373,0.2483,120,1229,-1.6288,0.1527,-1.6335,0.1710,-1.3384,0.4496,0.7461,0.4657,7.2079,801.0000,NaN,0.6841,-0.1671
2011-01-04,1562,0.2000,0.5904,0.1603,108,1454,-1.5189,0.1772,-1.6137,0.1755,-0.2630,0.6071,-0.3896,0.2843,7.3544,"1,349.0000",NaN,0.1579,0.0036
2011-01-05,1600,0.2270,0.4370,0.1869,82,1518,-1.4992,0.1815,-1.4664,0.2091,-1.3406,0.4493,-0.0463,0.3391,7.3784,"1,562.0000",NaN,0.0243,0.0270
2011-01-06,1606,0.2043,0.5183,0.0896,88,1518,-1.4961,0.1822,-1.5899,0.1809,-0.7697,0.5329,-1.3022,0.1385,7.3821,"1,600.0000",NaN,0.0037,-0.0226
2011-01-07,1510,0.1965,0.4987,0.1687,148,1362,-1.5457,0.1712,-1.6327,0.1712,-0.9071,0.5128,-0.2808,0.3017,7.3205,"1,606.0000",NaN,-0.0598,-0.0078
2011-01-08,959,0.1650,0.5358,0.2668,68,891,-1.8301,0.1078,-1.8049,0.1319,-0.6464,0.5510,0.9848,0.5039,6.8669,"1,510.0000",985.0000,-0.3649,-0.0315
2011-01-09,822,0.1383,0.4342,0.3619,54,768,-1.9009,0.0920,-1.9506,0.0987,-1.3602,0.4464,2.2125,0.7000,6.7130,959.0000,801.0000,-0.1429,-0.0267


## 4) 구조 변환: melt / pivot_table

전처리에서 가장 자주 막히는 지점이 long/wide 변환입니다.  
한 번 정리해두면 시각화/집계/모델 입력을 모두 같은 원리로 처리할 수 있습니다.

### 변환 목적

- `melt`: 변수 축을 세로로 눕혀 범용 집계 코드 작성
- `pivot_table`: 보고서용/모델용 매트릭스로 다시 펼치기

아래 코드에서는 월별 데이터 기준으로 변환하며, 연-월 × 변수 매트릭스를 만듭니다.

In [50]:
monthly_ts

,cnt,casual,registered,cnt_ma3,casual_share
date,,,,,
2011-01-31,38189,3073,35116,"38,189.0000",0.0805
2011-02-28,48215,6242,41973,"43,202.0000",0.1295
2011-03-31,64045,12826,51219,"50,149.6667",0.2003
2011-04-30,94870,22346,72524,"69,043.3333",0.2355
2011-05-31,135821,31050,104771,"98,245.3333",0.2286
2011-06-30,143512,30612,112900,"124,734.3333",0.2133
2011-07-31,141341,36452,104889,"140,224.6667",0.2579
2011-08-31,136691,28842,107849,"140,514.6667",0.2110
2011-09-30,127418,26545,100873,"135,150.0000",0.2083


In [59]:
monthly_ts.shape

(24, 5)

In [51]:
monthly_reset = monthly_ts.reset_index()
monthly_reset["year"] = monthly_reset["date"].dt.year
monthly_reset["month"] = monthly_reset["date"].dt.month

monthly_reset.head(8)

,date,cnt,casual,registered,cnt_ma3,casual_share,year,month
0,2011-01-31,38189,3073,35116,"38,189.0000",0.0805,2011,1
1,2011-02-28,48215,6242,41973,"43,202.0000",0.1295,2011,2
2,2011-03-31,64045,12826,51219,"50,149.6667",0.2003,2011,3
3,2011-04-30,94870,22346,72524,"69,043.3333",0.2355,2011,4
4,2011-05-31,135821,31050,104771,"98,245.3333",0.2286,2011,5
5,2011-06-30,143512,30612,112900,"124,734.3333",0.2133,2011,6
6,2011-07-31,141341,36452,104889,"140,224.6667",0.2579,2011,7
7,2011-08-31,136691,28842,107849,"140,514.6667",0.2110,2011,8


In [55]:
monthly_long = monthly_reset.melt(
    id_vars=["date", "year", "month"],
    value_vars=["cnt", "casual", "registered", "cnt_ma3", "casual_share"],
    var_name="metric",
    value_name="value"
)

print("[monthly_long head]")
display(monthly_long.head(100))

print("[monthly_long shape]")
print(monthly_long.shape)

[monthly_long head]


,date,year,month,metric,value
0,2011-01-31,2011,1,cnt,"38,189.0000"
1,2011-02-28,2011,2,cnt,"48,215.0000"
2,2011-03-31,2011,3,cnt,"64,045.0000"
3,2011-04-30,2011,4,cnt,"94,870.0000"
4,2011-05-31,2011,5,cnt,"135,821.0000"
...,...,...,...,...,...
95,2012-12-31,2012,12,cnt_ma3,"158,406.0000"
96,2011-01-31,2011,1,casual_share,0.0805
97,2011-02-28,2011,2,casual_share,0.1295
98,2011-03-31,2011,3,casual_share,0.2003


[monthly_long shape]
(120, 5)


In [58]:
monthly_pivot = monthly_long.pivot_table(
    index=["year", "month"],
    columns="metric", #이미 있는 칼럼
    values="value", # 이름 지을 칼럼
    aggfunc="mean"
).reset_index()

print("[monthly_pivot head]")
display(monthly_pivot.head(24))

print("[monthly_pivot shape]")
print(monthly_pivot.shape)

[monthly_pivot head]


metric,year,month,casual,casual_share,cnt,cnt_ma3,registered
0,2011,1,"3,073.0000",0.0805,"38,189.0000","38,189.0000","35,116.0000"
1,2011,2,"6,242.0000",0.1295,"48,215.0000","43,202.0000","41,973.0000"
2,2011,3,"12,826.0000",0.2003,"64,045.0000","50,149.6667","51,219.0000"
3,2011,4,"22,346.0000",0.2355,"94,870.0000","69,043.3333","72,524.0000"
4,2011,5,"31,050.0000",0.2286,"135,821.0000","98,245.3333","104,771.0000"
5,2011,6,"30,612.0000",0.2133,"143,512.0000","124,734.3333","112,900.0000"
6,2011,7,"36,452.0000",0.2579,"141,341.0000","140,224.6667","104,889.0000"
7,2011,8,"28,842.0000",0.2110,"136,691.0000","140,514.6667","107,849.0000"
8,2011,9,"26,545.0000",0.2083,"127,418.0000","135,150.0000","100,873.0000"
9,2011,10,"25,222.0000",0.2042,"123,511.0000","129,206.6667","98,289.0000"


[monthly_pivot shape]
(24, 7)


In [21]:
print(len(monthly_long))
print(len(monthly_pivot))

120
24


## 5) 다변량 입력 포맷 만들기

다변량 시계열의 **입력 매트릭스 형태**를 만들어 보겠습니다.

- 인덱스: 시간축(`date`)
- 컬럼: 다변량 수치 피처
- 결측: lag/차분에서 발생한 앞부분 결측 제거

### 생각해보기

1. `resample("ME")` 대신 `resample("MS")`를 써서 월초 기준으로 바꿔보세요.
2. `agg`에 `median`, `max`, `min`을 추가해 월별 요약표를 확장하세요.
3. `weather_name`별 월평균 수요를 `pivot_table`로 만들어 비교하세요.
4. `cnt`, `temp`, `hum`의 월별 상관관계를 연도별로 계산해 보세요.
5. 결측 처리 방식(`dropna`, `ffill`)에 따라 결과가 어떻게 달라지는지 비교하세요.

In [22]:
# 다변량 시계열 입력 매트릭스 생성 (예시)
# 본 셀은 입력 구조를 연습하기 위한 예시입니다.

var_like_input = (
    engineered[["cnt", "temp", "hum", "windspeed", "cnt_pct_change", "temp_diff1"]]
    .dropna()
    .copy()
)

print("var_like_input shape:", var_like_input.shape)
print("index monotonic increasing:", var_like_input.index.is_monotonic_increasing)
print("\n상관계수")
display(var_like_input.corr().round(3))

display(var_like_input.head())

var_like_input shape: (730, 6)
index monotonic increasing: True

상관계수


,cnt,temp,hum,windspeed,cnt_pct_change,temp_diff1
cnt,1.0000,0.6270,-0.0980,-0.2360,-0.0440,0.1340
temp,0.6270,1.0000,0.1290,-0.1580,-0.0540,0.1630
hum,-0.0980,0.1290,1.0000,-0.2480,0.0250,0.0810
windspeed,-0.2360,-0.1580,-0.2480,1.0000,0.0090,-0.1690
cnt_pct_change,-0.0440,-0.0540,0.0250,0.0090,1.0000,-0.0370
temp_diff1,0.1340,0.1630,0.0810,-0.1690,-0.0370,1.0000


,cnt,temp,hum,windspeed,cnt_pct_change,temp_diff1
date,,,,,,
2011-01-02,801,0.3635,0.6961,0.2485,-0.1868,0.0193
2011-01-03,1349,0.1964,0.4373,0.2483,0.6841,-0.1671
2011-01-04,1562,0.2000,0.5904,0.1603,0.1579,0.0036
2011-01-05,1600,0.2270,0.4370,0.1869,0.0243,0.0270
2011-01-06,1606,0.2043,0.5183,0.0896,0.0037,-0.0226
